# Week 8 - LLM-First Agentic Deal Orchestration

This notebook is refactored to be **LLM-first**:
1. LLM valuation agent
2. LLM decision agent
3. Agent orchestrator class
4. Explainable deal cards
5. Feedback loop that influences subsequent runs

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional
import hashlib
import json
import os

## 1) Schemas and core helpers

In [ ]:
FeedbackType = Literal["upvote", "downvote", "skip"]
ActionType = Literal["alert_user", "watchlist", "skip"]


@dataclass
class DealPrediction:
    title: str
    url: str
    listed_price: float
    estimate_specialist: float
    estimate_frontier: float
    retrieval_scores: List[float]
    comparables: List[dict]


@dataclass
class FeedbackRecord:
    deal_id: str
    feedback: FeedbackType
    timestamp: str
    notes: str = ""


@dataclass
class ValuationOutput:
    estimated_value: float
    confidence: float
    rationale: str
    risk_flags: List[str]


@dataclass
class DealCard:
    deal_id: str
    title: str
    url: str
    listed_price: float
    estimated_fair_value: float
    discount: float
    confidence_score: float
    confidence_bucket: Literal["high", "medium", "low"]
    rationale: str
    risk_flags: List[str]
    retrieved_comparables: List[dict]
    agent_score: float


@dataclass
class AgentDecision:
    action: ActionType
    selected_deal_id: str
    rationale: str
    ranked_cards: List[DealCard]


def stable_deal_id(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()[:16]


def clamp(value: float, lo: float = 0.0, hi: float = 1.0) -> float:
    return max(lo, min(value, hi))


def confidence_bucket(score: float) -> Literal["high", "medium", "low"]:
    if score >= 0.75:
        return "high"
    if score >= 0.5:
        return "medium"
    return "low"

## 2) Feedback memory

In [ ]:
FEEDBACK_PATH = Path("feedback_log.json")


def load_feedback(path: Path = FEEDBACK_PATH) -> List[FeedbackRecord]:
    if not path.exists():
        return []
    raw = json.loads(path.read_text(encoding="utf-8"))
    return [FeedbackRecord(**item) for item in raw]


def save_feedback(records: List[FeedbackRecord], path: Path = FEEDBACK_PATH) -> None:
    payload = [r.__dict__ for r in records]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def append_feedback(
    deal_id: str,
    feedback: FeedbackType,
    notes: str = "",
    path: Path = FEEDBACK_PATH,
) -> FeedbackRecord:
    records = load_feedback(path)
    record = FeedbackRecord(
        deal_id=deal_id,
        feedback=feedback,
        notes=notes,
        timestamp=datetime.now(timezone.utc).isoformat(),
    )
    records.append(record)
    save_feedback(records, path)
    return record


def feedback_index(records: List[FeedbackRecord]) -> Dict[str, List[FeedbackRecord]]:
    index: Dict[str, List[FeedbackRecord]] = {}
    for record in records:
        index.setdefault(record.deal_id, []).append(record)
    return index


def feedback_prior_for_deal(deal_id: str, index: Dict[str, List[FeedbackRecord]]) -> float:
    records = index.get(deal_id, [])
    if not records:
        return 0.0
    score_map = {"upvote": 1.0, "downvote": -1.0, "skip": 0.0}
    avg = sum(score_map[r.feedback] for r in records) / len(records)
    return 0.1 * avg

## 3) LLM agent tools

In [ ]:
def get_openai_client():
    try:
        from openai import OpenAI
    except ImportError as exc:
        raise ImportError("Install OpenAI SDK first: pip install openai") from exc

    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Missing OPENAI_API_KEY in environment.")

    return OpenAI()


def llm_json(client, model: str, system_prompt: str, user_prompt: str) -> Dict[str, Any]:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )
    content = response.choices[0].message.content
    return json.loads(content)


VALUATION_SYSTEM_PROMPT = """You are a valuation agent for online product deals.
Return strict JSON only with keys:
- estimated_value (number)
- confidence (number 0 to 1)
- rationale (string, max 2 sentences)
- risk_flags (array of short strings)
Use listed price, retrieval scores, and comparable prices as evidence.
"""

DECISION_SYSTEM_PROMPT = """You are a decision agent for deals.
Return strict JSON only with keys:
- action (one of: alert_user, watchlist, skip)
- selected_deal_id (string)
- rationale (string, max 2 sentences)
Choose alert_user only when both confidence and discount are strong.
"""

## 4) LLM-first orchestrator class

In [ ]:
class LLMFirstDealOrchestrator:
    def __init__(
        self,
        valuation_model: str = "gpt-4.1-mini",
        decision_model: str = "gpt-4.1-mini",
    ):
        self.valuation_model = valuation_model
        self.decision_model = decision_model
        self.client = get_openai_client()

    def valuate(self, deal: DealPrediction, feedback_prior: float = 0.0) -> ValuationOutput:
        payload = {
            "title": deal.title,
            "url": deal.url,
            "listed_price": deal.listed_price,
            "estimate_specialist": deal.estimate_specialist,
            "estimate_frontier": deal.estimate_frontier,
            "retrieval_scores": deal.retrieval_scores,
            "comparables": deal.comparables,
            "feedback_prior": feedback_prior,
        }
        result = llm_json(
            client=self.client,
            model=self.valuation_model,
            system_prompt=VALUATION_SYSTEM_PROMPT,
            user_prompt=f"Valuate this deal: {json.dumps(payload)}",
        )
        return ValuationOutput(
            estimated_value=float(result.get("estimated_value", 0.0)),
            confidence=clamp(float(result.get("confidence", 0.0))),
            rationale=str(result.get("rationale", "No rationale provided.")),
            risk_flags=[str(x) for x in result.get("risk_flags", [])],
        )

    @staticmethod
    def _opportunity_score(card: DealCard) -> float:
        discount_component = clamp(card.discount / max(card.estimated_fair_value, 1e-6))
        return round(0.7 * card.confidence_score + 0.3 * discount_component, 4)

    def to_card(self, deal: DealPrediction, valuation: ValuationOutput) -> DealCard:
        estimated = round(valuation.estimated_value, 2)
        discount = round(estimated - deal.listed_price, 2)
        conf = round(clamp(valuation.confidence), 4)
        top_comps = sorted(
            deal.comparables,
            key=lambda c: float(c.get("similarity", 0.0)),
            reverse=True,
        )[:3]
        card = DealCard(
            deal_id=stable_deal_id(deal.url),
            title=deal.title,
            url=deal.url,
            listed_price=round(deal.listed_price, 2),
            estimated_fair_value=estimated,
            discount=discount,
            confidence_score=conf,
            confidence_bucket=confidence_bucket(conf),
            rationale=valuation.rationale,
            risk_flags=valuation.risk_flags,
            retrieved_comparables=top_comps,
            agent_score=0.0,
        )
        card.agent_score = self._opportunity_score(card)
        return card

    def decide(self, ranked_cards: List[DealCard]) -> AgentDecision:
        compact = [
            {
                "deal_id": c.deal_id,
                "title": c.title,
                "listed_price": c.listed_price,
                "estimated_fair_value": c.estimated_fair_value,
                "discount": c.discount,
                "confidence_score": c.confidence_score,
                "risk_flags": c.risk_flags,
                "agent_score": c.agent_score,
            }
            for c in ranked_cards
        ]
        result = llm_json(
            client=self.client,
            model=self.decision_model,
            system_prompt=DECISION_SYSTEM_PROMPT,
            user_prompt=f"Choose the best action for these ranked deals: {json.dumps(compact)}",
        )
        action = str(result.get("action", "skip"))
        if action not in {"alert_user", "watchlist", "skip"}:
            action = "skip"
        selected_deal_id = str(result.get("selected_deal_id", ""))
        if not selected_deal_id and ranked_cards:
            selected_deal_id = ranked_cards[0].deal_id
        return AgentDecision(
            action=action,
            selected_deal_id=selected_deal_id,
            rationale=str(result.get("rationale", "No rationale provided.")),
            ranked_cards=ranked_cards,
        )

    def run(self, deals: List[DealPrediction]) -> AgentDecision:
        index = feedback_index(load_feedback())
        cards: List[DealCard] = []
        for deal in deals:
            prior = feedback_prior_for_deal(stable_deal_id(deal.url), index)
            valuation = self.valuate(deal, feedback_prior=prior)
            cards.append(self.to_card(deal, valuation))
        cards.sort(key=lambda c: c.agent_score, reverse=True)
        return self.decide(cards)

## 5) Sample deal inputs

In [ ]:
sample_1 = DealPrediction(
    title="Sony WH-1000XM5 Wireless Noise-Canceling Headphones",
    url="https://example.com/deals/sony-xm5",
    listed_price=249.99,
    estimate_specialist=329.0,
    estimate_frontier=309.0,
    retrieval_scores=[0.82, 0.78, 0.74],
    comparables=[
        {"title": "Sony WH-1000XM5 Black", "price": 329.0, "similarity": 0.88},
        {"title": "Sony WH-1000XM4", "price": 279.0, "similarity": 0.81},
        {"title": "Bose QC Ultra", "price": 349.0, "similarity": 0.72},
    ],
)

sample_2 = DealPrediction(
    title="Apple AirPods Pro (2nd Gen)",
    url="https://example.com/deals/airpods-pro-2",
    listed_price=219.99,
    estimate_specialist=239.0,
    estimate_frontier=231.0,
    retrieval_scores=[0.66, 0.61, 0.58],
    comparables=[
        {"title": "AirPods Pro 2 USB-C", "price": 249.0, "similarity": 0.74},
        {"title": "AirPods Pro 2 Lightning", "price": 229.0, "similarity": 0.69},
        {"title": "Galaxy Buds Pro", "price": 179.0, "similarity": 0.52},
    ],
)

sample_3 = DealPrediction(
    title="Generic Budget Earbuds",
    url="https://example.com/deals/generic-buds",
    listed_price=89.99,
    estimate_specialist=92.0,
    estimate_frontier=145.0,
    retrieval_scores=[0.41, 0.35, 0.33],
    comparables=[
        {"title": "Budget Earbuds A", "price": 39.0, "similarity": 0.42},
        {"title": "Budget Earbuds B", "price": 49.0, "similarity": 0.37},
    ],
)

candidates = [sample_1, sample_2, sample_3]
print(f"Prepared {len(candidates)} candidate deals")

## 6) Run LLM-first orchestration

In [ ]:
# Requires OPENAI_API_KEY in your environment.
if os.getenv("OPENAI_API_KEY"):
    orchestrator = LLMFirstDealOrchestrator(
        valuation_model="gpt-4.1-mini",
        decision_model="gpt-4.1-mini",
    )
    decision = orchestrator.run(candidates)
    print("ACTION:", decision.action)
    print("SELECTED DEAL ID:", decision.selected_deal_id)
    print("RATIONALE:", decision.rationale)
    print("\nRANKED CARDS:")
    for i, card in enumerate(decision.ranked_cards, start=1):
        print(i, card.title, "score=", card.agent_score, "confidence=", card.confidence_score, "discount=", card.discount)
else:
    print("OPENAI_API_KEY missing. Skip this cell and run the fallback cell below.")

In [ ]:
if "decision" in locals():
    selected = next((c for c in decision.ranked_cards if c.deal_id == decision.selected_deal_id), None)
    if selected:
        print("\nSELECTED CARD JSON:")
        print(json.dumps(selected.__dict__, indent=2))
    else:
        print("No selected card found.")
else:
    print("No LLM decision object yet.")

## 7) Feedback writeback and second run

In [ ]:
# Example: user feedback after reviewing selected deal
if "selected" in locals() and selected:
    append_feedback(
        deal_id=selected.deal_id,
        feedback="upvote",
        notes="Good balance of confidence and discount.",
    )

if "orchestrator" in locals():
    # Re-run orchestration to include feedback prior
    second_decision = orchestrator.run(candidates)
    print("SECOND RUN ACTION:", second_decision.action)
    print("SECOND RUN RATIONALE:", second_decision.rationale)
else:
    print("No LLM orchestrator initialized yet.")

In [ ]:
# If you need to inspect feedback file quickly:
print("Feedback records:", len(load_feedback()))
for row in load_feedback()[-3:]:
    print(row)

In [ ]:
# Minimal deterministic fallback if API key is missing.
# This keeps the notebook runnable even without external API access.

def fallback_decision(deals: List[DealPrediction]) -> AgentDecision:
    cards: List[DealCard] = []
    for deal in deals:
        fair = (deal.estimate_specialist + deal.estimate_frontier) / 2.0
        discount = fair - deal.listed_price
        conf = clamp(sum(deal.retrieval_scores) / max(len(deal.retrieval_scores), 1))
        card = DealCard(
            deal_id=stable_deal_id(deal.url),
            title=deal.title,
            url=deal.url,
            listed_price=deal.listed_price,
            estimated_fair_value=round(fair, 2),
            discount=round(discount, 2),
            confidence_score=round(conf, 4),
            confidence_bucket=confidence_bucket(conf),
            rationale="Fallback path used because OPENAI_API_KEY is unavailable.",
            risk_flags=[],
            retrieved_comparables=deal.comparables[:3],
            agent_score=round(0.7 * conf + 0.3 * clamp(discount / max(fair, 1e-6)), 4),
        )
        cards.append(card)

    cards.sort(key=lambda c: c.agent_score, reverse=True)
    selected_id = cards[0].deal_id if cards else ""
    action: ActionType = "alert_user" if cards and cards[0].confidence_score >= 0.7 and cards[0].discount > 30 else "watchlist"
    return AgentDecision(
        action=action,
        selected_deal_id=selected_id,
        rationale="Fallback deterministic decision.",
        ranked_cards=cards,
    )

In [ ]:
# Unified run: prefer LLM orchestration, fallback if key is missing.
if os.getenv("OPENAI_API_KEY"):
    final_decision = orchestrator.run(candidates)
else:
    final_decision = fallback_decision(candidates)

print("FINAL ACTION:", final_decision.action)
print("FINAL RATIONALE:", final_decision.rationale)
print("FINAL SELECTED:", final_decision.selected_deal_id)

selected_final = next((c for c in final_decision.ranked_cards if c.deal_id == final_decision.selected_deal_id), None)
if selected_final:
    print(json.dumps(selected_final.__dict__, indent=2))